# Motion-crop bird detector: stride-16 grid, ignore mask, and split outputs

This notebook is a new, incompatible successor to `train_bird_detector_motion_crop_tflite.ipynb`.

Changes in this version:

* adds an `ignore` target channel; every non-positive slot whose **cell centre** is inside a bird box is excluded from the negative score loss;
* uses MobileNetV2 `block_13_expand_relu` (stride 16) instead of `block_6_expand_relu` (stride 8);
* defaults to a cheaper 224×224 input (14×14 grid), while 192×192 (12×12) and 288×288 (18×18) remain available;
* re-fits anchors for the new configuration;
* exports boxes and bird-score logits as separate TFLite output tensors, each produced by its own 1×1 convolution;
* compares full-integer UINT8 inference against float inference on every run;
* selects the deployment threshold only from the UINT8 model's validation sweep.

The old notebook, anchors, checkpoints, and exports are not modified or reused.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator, Mapping, Sequence

import numpy as np
import pandas as pd
from PIL import Image


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "Models" / "Multi_Object_Detector" / "dataset_dirs.txt").is_file():
            return candidate
    raise FileNotFoundError("Could not find Models/Multi_Object_Detector/dataset_dirs.txt")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Models.Multi_Object_Detector import train_bird_detector_tflite as base

print("Project:", PROJECT_ROOT)


Project: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier


## Configuration

`RUN_TRAINING=False` requires compatible final weights in this notebook's own artifact directory. Export, float/UINT8 drift comparison, and validation threshold selection still run every time the later cells are executed.


In [2]:
SOURCE_RUN = ""  # completed source-run name or absolute path; empty selects the newest
INPUT_SIZE = 224  # 192 -> 12x12, 224 -> 14x14, 288 -> 18x18
BATCH_SIZE = 8
VERIFY_IMAGES = True
RUN_TRAINING = True
REFIT_ANCHORS = True

ALLOWED_INPUT_SIZES = (192, 224, 288)
FEATURE_LAYER = "block_13_expand_relu"
FEATURE_STRIDE = 16
BACKBONE_ALPHA = base.BACKBONE_ALPHA
NUM_ANCHORS = base.NUM_ANCHORS
NEIGHBOUR_RADIUS = base.NEIGHBOUR_RADIUS
CENTER_DECODE_SPAN_CELLS = base.CENTER_DECODE_SPAN_CELLS

TARGET_BOX_SLICE = slice(0, 4)
TARGET_OBJECTNESS = 4
TARGET_IGNORE = 5
TARGET_BIRD_COUNT = 6
TARGET_CHANNELS = 7

if INPUT_SIZE not in ALLOWED_INPUT_SIZES:
    raise ValueError(f"INPUT_SIZE must be one of {ALLOWED_INPUT_SIZES}")
if INPUT_SIZE % FEATURE_STRIDE:
    raise ValueError("INPUT_SIZE must be divisible by FEATURE_STRIDE")

GRID_SIZE = INPUT_SIZE // FEATURE_STRIDE
print(f"Input: {INPUT_SIZE}x{INPUT_SIZE}; feature layer: {FEATURE_LAYER}; grid: {GRID_SIZE}x{GRID_SIZE}")


Input: 224x224; feature layer: block_13_expand_relu; grid: 14x14


## Load and audit crop-only datasets

Artifacts are isolated under `keras_motion_crop_detector_stride16_split_heads_<input-size>`. This deliberately prevents old stride-8 weights and anchors from being loaded.


In [3]:
random.seed(base.SEED)
np.random.seed(base.SEED)

source_run = base.resolve_source_run(PROJECT_ROOT, SOURCE_RUN or None)
artifact_dir = source_run / f"keras_motion_crop_detector_stride16_split_heads_{INPUT_SIZE}"
checkpoint_dir = artifact_dir / "checkpoints"
export_dir = artifact_dir / "exports"
for directory in (artifact_dir, checkpoint_dir, export_dir):
    directory.mkdir(parents=True, exist_ok=True)

paths = base.RunPaths(
    source_run=source_run,
    artifact_dir=artifact_dir,
    checkpoint_dir=checkpoint_dir,
    export_dir=export_dir,
    head_weights=checkpoint_dir / "head.weights.h5",
    final_weights=checkpoint_dir / "final.weights.h5",
    anchors=artifact_dir / "anchors.json",
    float_tflite=export_dir / "bird_detector_motion_crop_stride16_float32.tflite",
    uint8_tflite=export_dir / "bird_detector_motion_crop_stride16_uint8.tflite",
)

scenes = base.load_scene_records(source_run / "joined_bird_detector_annotations.csv")
datasets = base.select_datasets(scenes)
audit = base.audit_datasets(datasets, verify_images=VERIFY_IMAGES)
audit.to_csv(artifact_dir / "crop_dataset_audit.csv", index=False)
display(audit)

if VERIFY_IMAGES and audit.loc[
    audit.dataset.isin(["train_crop", "validation_crop"]), "missing_images"
].sum():
    raise FileNotFoundError("Training or validation images are missing; see crop_dataset_audit.csv")

print("Source run:", source_run)
print("New artifact directory:", artifact_dir)


,dataset,images,positive_images,negative_images,birds,missing_images
0,train_crop,12677,10065,2612,11617,0
1,validation_crop,506,373,133,500,0
2,test_crop,147,80,67,112,0
3,calibration_crop,500,250,250,330,0


Source run: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z
New artifact directory: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z/keras_motion_crop_detector_stride16_split_heads_224


## Target encoding with bird-interior ignore slots

Positive assignment is unchanged: a bird is assigned to its centre cell and valid neighbouring cells, using the best free anchor in each cell. After all positives are assigned, every remaining slot is marked `ignore=1` when that slot's cell centre lies inside any bird box. Positive slots always have `ignore=0`.


In [4]:
def encode_targets_numpy(
    boxes: np.ndarray,
    anchors: np.ndarray,
    grid_size: int,
    neighbour_radius: int = NEIGHBOUR_RADIUS,
) -> np.ndarray:
    """Encode boxes plus an ignore channel for unassigned bird-interior slots."""
    boxes = np.asarray(boxes, dtype=np.float32).reshape((-1, 4))
    anchors = np.asarray(anchors, dtype=np.float32)
    target = np.zeros((grid_size, grid_size, len(anchors), TARGET_CHANNELS), dtype=np.float32)
    target[..., TARGET_BIRD_COUNT] = float(len(boxes))
    if not len(boxes):
        return target

    widths = np.maximum(boxes[:, 2] - boxes[:, 0], 1e-6)
    heights = np.maximum(boxes[:, 3] - boxes[:, 1], 1e-6)
    centers_x = np.clip((boxes[:, 0] + boxes[:, 2]) / 2, 0, 1 - 1e-6)
    centers_y = np.clip((boxes[:, 1] + boxes[:, 3]) / 2, 0, 1 - 1e-6)
    normalized_boxes = np.stack([centers_x, centers_y, widths, heights], axis=-1)
    anchor_ious = base.wh_iou(normalized_boxes[:, 2:4], anchors)

    # Assign the hardest-to-place birds first, then use the best free anchor per cell.
    bird_order = np.argsort(anchor_ious.max(axis=1))[::-1]
    for bird_index in bird_order:
        center_cell_x = int(math.floor(float(centers_x[bird_index]) * grid_size))
        center_cell_y = int(math.floor(float(centers_y[bird_index]) * grid_size))
        ranked_anchors = np.argsort(anchor_ious[bird_index])[::-1]
        for dy in range(-neighbour_radius, neighbour_radius + 1):
            for dx in range(-neighbour_radius, neighbour_radius + 1):
                cell_x, cell_y = center_cell_x + dx, center_cell_y + dy
                if not (0 <= cell_x < grid_size and 0 <= cell_y < grid_size):
                    continue
                free = [
                    anchor for anchor in ranked_anchors
                    if target[cell_y, cell_x, anchor, TARGET_OBJECTNESS] < 0.5
                ]
                if not free:
                    continue
                anchor = int(free[0])
                target[cell_y, cell_x, anchor, TARGET_BOX_SLICE] = normalized_boxes[bird_index]
                target[cell_y, cell_x, anchor, TARGET_OBJECTNESS] = 1.0

    # Cell centres, not predicted anchor boxes, define the ignored bird interior.
    cell_centres = (np.arange(grid_size, dtype=np.float32) + 0.5) / float(grid_size)
    cell_x, cell_y = np.meshgrid(cell_centres, cell_centres, indexing="xy")
    inside_any_bird = np.zeros((grid_size, grid_size), dtype=bool)
    for xmin, ymin, xmax, ymax in boxes:
        inside_any_bird |= (
            (cell_x >= xmin) & (cell_x <= xmax) &
            (cell_y >= ymin) & (cell_y <= ymax)
        )
    target[..., TARGET_IGNORE][inside_any_bird, :] = 1.0
    target[..., TARGET_IGNORE] *= 1.0 - target[..., TARGET_OBJECTNESS]
    return target


# Fast invariants before TensorFlow is involved.
_test_anchors = np.asarray([[0.15, 0.15], [0.30, 0.30], [0.60, 0.60]], np.float32)
_test_target = encode_targets_numpy(
    np.asarray([[0.25, 0.25, 0.75, 0.75]], np.float32), _test_anchors, grid_size=4
)
assert not np.any(
    (_test_target[..., TARGET_OBJECTNESS] > 0.5) &
    (_test_target[..., TARGET_IGNORE] > 0.5)
)
assert int(_test_target[..., TARGET_IGNORE].sum()) > 0
assert np.all(_test_target[..., TARGET_BIRD_COUNT] == 1)
print("Encoder checks passed; ignored slots:", int(_test_target[..., TARGET_IGNORE].sum()))


Encoder checks passed; ignored slots: 8


## Re-fit anchors for the new model contract

`REFIT_ANCHORS=True` is the default. Anchor fitting is deterministic, and the saved fingerprint includes the feature layer, stride, input size, target layout, split-head contract, and ignore policy.


In [5]:
fingerprint_payload = {
    **base.configuration_fingerprint_payload(
        base.sha256_file(source_run / "joined_bird_detector_annotations.csv"), INPUT_SIZE
    ),
    "feature_layer": FEATURE_LAYER,
    "feature_stride": FEATURE_STRIDE,
    "grid_size": GRID_SIZE,
    "target_channels": ["cx", "cy", "width", "height", "objectness", "ignore", "bird_count"],
    "ignore_policy": "non-positive slots ignored when their cell centre is inside any bird box",
    "head": "separate 1x1 box and score convolutions",
    "outputs": {"box_output": 4, "score_output": 1},
}
fingerprint = hashlib.sha256(
    json.dumps(base.json_safe(fingerprint_payload), sort_keys=True).encode("utf-8")
).hexdigest()

configuration_path = artifact_dir / "configuration.json"
if configuration_path.is_file():
    existing = json.loads(configuration_path.read_text(encoding="utf-8"))
    if existing.get("config_fingerprint") != fingerprint:
        raise RuntimeError(
            f"Incompatible artifacts already exist in {artifact_dir}. Archive or remove that directory."
        )
else:
    configuration_path.write_text(
        json.dumps({
            "config_fingerprint": fingerprint,
            "configuration": base.json_safe(fingerprint_payload),
        }, indent=2),
        encoding="utf-8",
    )

if REFIT_ANCHORS or not paths.anchors.is_file():
    anchors = base.fit_anchors(
        base.normalized_letterbox_wh(datasets["train_crop"], INPUT_SIZE), NUM_ANCHORS
    )
    paths.anchors.write_text(
        json.dumps({
            "config_fingerprint": fingerprint,
            "input_size": INPUT_SIZE,
            "feature_layer": FEATURE_LAYER,
            "feature_stride": FEATURE_STRIDE,
            "grid_size": GRID_SIZE,
            "anchors": anchors.tolist(),
        }, indent=2),
        encoding="utf-8",
    )
    print("Re-fitted anchors for the stride-16 crop model.")
else:
    anchor_contract = json.loads(paths.anchors.read_text(encoding="utf-8"))
    if anchor_contract.get("config_fingerprint") != fingerprint:
        raise RuntimeError("Saved anchors belong to a different configuration")
    anchors = np.asarray(anchor_contract["anchors"], np.float32)

display(pd.DataFrame(anchors, columns=["width", "height"]))


Re-fitted anchors for the stride-16 crop model.


,width,height
0,0.301974,0.306620
1,0.502787,0.552846
2,0.831633,0.674772


## Split-head model, dataset, and ignore-aware losses

The box and score branches each end in their own 1×1 convolution. The score loss's negative mask is `(1 - positive) * (1 - ignore)`, so the unassigned remainder of a bird is neither a positive nor a negative. The box loss is positive-only.


In [6]:
tf = base.require_tensorflow()
tf.random.set_seed(base.SEED)


def build_model(tf: Any, input_size: int) -> tuple[Any, Any, int]:
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(input_size, input_size, 3),
        alpha=BACKBONE_ALPHA,
        include_top=False,
        weights="imagenet",
    )
    feature_extractor = tf.keras.Model(
        backbone.input,
        backbone.get_layer(FEATURE_LAYER).output,
        name="mobilenetv2_block13_finder",
    )
    inputs = tf.keras.Input((input_size, input_size, 3), name="image", dtype=tf.float32)
    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1, name="normalize_uint8_rgb")(inputs)
    x = feature_extractor(x)
    x = tf.keras.layers.SeparableConv2D(
        96, 3, padding="same", use_bias=False, name="finder_sepconv"
    )(x)
    x = tf.keras.layers.BatchNormalization(name="finder_bn")(x)
    x = tf.keras.layers.ReLU(max_value=6.0, name="finder_relu6")(x)

    box_logits = tf.keras.layers.Conv2D(
        NUM_ANCHORS * 4, 1, padding="same", name="box_logits_conv"
    )(x)
    score_logits = tf.keras.layers.Conv2D(
        NUM_ANCHORS, 1, padding="same", name="score_logits_conv"
    )(x)
    grid_height, grid_width = int(box_logits.shape[1]), int(box_logits.shape[2])
    if grid_height != grid_width or grid_height != input_size // FEATURE_STRIDE:
        raise ValueError(
            f"{FEATURE_LAYER} produced {grid_height}x{grid_width}; expected "
            f"{input_size // FEATURE_STRIDE}x{input_size // FEATURE_STRIDE}"
        )
    box_output = tf.keras.layers.Reshape(
        (grid_height, grid_width, NUM_ANCHORS, 4), name="box_output"
    )(box_logits)
    score_output = tf.keras.layers.Reshape(
        (grid_height, grid_width, NUM_ANCHORS, 1), name="score_output"
    )(score_logits)
    model = tf.keras.Model(
        inputs=inputs,
        outputs={"box_output": box_output, "score_output": score_output},
        name="bird_detector_motion_crop_stride16_split_heads",
    )
    return model, feature_extractor, grid_height


def make_dataset(
    records: Sequence[Mapping[str, Any]],
    anchors: np.ndarray,
    grid_size: int,
    input_size: int,
    batch_size: int,
    training: bool,
) -> Any:
    def generator() -> Iterator[tuple[str, np.ndarray, np.ndarray]]:
        for record in records:
            bounds = record.get("crop_bounds", (0, 0, record["width"], record["height"]))
            yield (
                str(record["image_path"]),
                np.asarray(bounds, np.int32),
                np.asarray(record["boxes"], np.float32),
            )

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec([], tf.string),
            tf.TensorSpec([4], tf.int32),
            tf.TensorSpec([None, 4], tf.float32),
        ),
    )
    if training:
        dataset = dataset.shuffle(
            min(len(records), 4096), seed=base.SEED, reshuffle_each_iteration=True
        )

    def load(path: Any, bounds: Any, boxes: Any) -> tuple[Any, Any]:
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        left, top, right, bottom = tf.unstack(bounds)
        image = tf.cast(image[top:bottom, left:right], tf.float32)
        height = tf.cast(tf.shape(image)[0], tf.float32)
        width = tf.cast(tf.shape(image)[1], tf.float32)
        scale = tf.minimum(input_size / width, input_size / height)
        new_width = tf.cast(tf.round(width * scale), tf.int32)
        new_height = tf.cast(tf.round(height * scale), tf.int32)
        resized = tf.image.resize(image, [new_height, new_width], antialias=True)
        pad_left = (input_size - new_width) // 2
        pad_top = (input_size - new_height) // 2
        image = tf.pad(
            resized,
            [
                [pad_top, input_size - new_height - pad_top],
                [pad_left, input_size - new_width - pad_left],
                [0, 0],
            ],
            constant_values=114.0,
        )
        image.set_shape([input_size, input_size, 3])
        vector = tf.cast(tf.stack([pad_left, pad_top, pad_left, pad_top]), tf.float32)
        boxes = tf.clip_by_value((boxes * scale + vector) / float(input_size), 0.0, 1.0)
        return image, boxes

    def augment(image: Any, boxes: Any) -> tuple[Any, Any]:
        flip = tf.random.uniform(()) < 0.5
        image = tf.cond(flip, lambda: tf.image.flip_left_right(image), lambda: image)
        boxes = tf.cond(
            flip,
            lambda: tf.stack(
                [1 - boxes[:, 2], boxes[:, 1], 1 - boxes[:, 0], boxes[:, 3]], axis=-1
            ),
            lambda: boxes,
        )
        image = tf.image.random_brightness(image, 12.0)
        image = tf.image.random_contrast(image, 0.85, 1.15)
        return tf.clip_by_value(image, 0.0, 255.0), boxes

    def encode(image: Any, boxes: Any) -> tuple[Any, dict[str, Any]]:
        target = tf.numpy_function(
            lambda value: encode_targets_numpy(value, anchors, grid_size),
            [boxes],
            tf.float32,
        )
        target.set_shape([grid_size, grid_size, NUM_ANCHORS, TARGET_CHANNELS])
        return image, {"box_output": target, "score_output": target}

    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return (
        dataset.map(encode, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


def compile_model(model: Any, anchors: np.ndarray, grid_size: int, learning_rate: float) -> None:
    anchor_tensor = tf.constant(anchors, tf.float32)[None, None, None, :, :]
    grid_y, grid_x = tf.meshgrid(
        tf.range(grid_size, dtype=tf.float32),
        tf.range(grid_size, dtype=tf.float32),
        indexing="ij",
    )
    grid = tf.stack([grid_x, grid_y], axis=-1)[None, :, :, None, :]
    huber = tf.keras.losses.Huber(reduction="none")

    def bird_normalizer(y_true: Any) -> Any:
        bird_counts = tf.reduce_max(y_true[..., TARGET_BIRD_COUNT], axis=[1, 2, 3])
        return tf.maximum(1.0, tf.reduce_sum(bird_counts))

    def decode_boxes(raw_box: Any) -> Any:
        center = (
            tf.sigmoid(raw_box[..., :2]) * CENTER_DECODE_SPAN_CELLS
            - (CENTER_DECODE_SPAN_CELLS - 1) / 2
            + grid
        ) / float(grid_size)
        size = anchor_tensor * tf.exp(tf.clip_by_value(raw_box[..., 2:4], -4.0, 4.0))
        return tf.concat([center, size], axis=-1)

    def box_loss(y_true: Any, raw_box: Any) -> Any:
        object_mask = y_true[..., TARGET_OBJECTNESS]
        coordinate_error = huber(y_true[..., TARGET_BOX_SLICE], decode_boxes(raw_box))
        return base.BOX_LOSS_WEIGHT * tf.reduce_sum(coordinate_error * object_mask) / bird_normalizer(y_true)

    def score_loss(y_true: Any, raw_score: Any) -> Any:
        object_mask = y_true[..., TARGET_OBJECTNESS]
        ignore_mask = y_true[..., TARGET_IGNORE]
        negative_mask = (1.0 - object_mask) * (1.0 - ignore_mask)
        logits = tf.squeeze(raw_score, axis=-1)
        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=object_mask, logits=logits)
        probability = tf.sigmoid(logits)
        probability_true = object_mask * probability + (1.0 - object_mask) * (1.0 - probability)
        focal = tf.square(1.0 - probability_true) * bce
        normalizer = bird_normalizer(y_true)
        positive = base.POSITIVE_OBJECTNESS_WEIGHT * tf.reduce_sum(focal * object_mask) / normalizer
        negative = base.NEGATIVE_OBJECTNESS_WEIGHT * tf.reduce_sum(focal * negative_mask) / normalizer
        return positive + negative

    def mean_positive_score(y_true: Any, raw_score: Any) -> Any:
        mask = y_true[..., TARGET_OBJECTNESS]
        probability = tf.sigmoid(tf.squeeze(raw_score, axis=-1))
        return tf.reduce_sum(probability * mask) / tf.maximum(1.0, tf.reduce_sum(mask))

    def mean_negative_score(y_true: Any, raw_score: Any) -> Any:
        mask = (1.0 - y_true[..., TARGET_OBJECTNESS]) * (1.0 - y_true[..., TARGET_IGNORE])
        probability = tf.sigmoid(tf.squeeze(raw_score, axis=-1))
        return tf.reduce_sum(probability * mask) / tf.maximum(1.0, tf.reduce_sum(mask))

    def mean_ignored_score(y_true: Any, raw_score: Any) -> Any:
        mask = y_true[..., TARGET_IGNORE]
        probability = tf.sigmoid(tf.squeeze(raw_score, axis=-1))
        return tf.reduce_sum(probability * mask) / tf.maximum(1.0, tf.reduce_sum(mask))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss={"box_output": box_loss, "score_output": score_loss},
        metrics={
            "score_output": [mean_positive_score, mean_negative_score, mean_ignored_score]
        },
    )


model, feature_extractor, model_grid_size = build_model(tf, INPUT_SIZE)
assert model_grid_size == GRID_SIZE
model.summary()


I0000 00:00:1793055474.670943    3647 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793055475.011799    3647 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793055476.784607    3647 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793055478.499521    3647 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB mem

Model: "bird_detector_motion_crop_stride16_split_heads"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_uint8_rgb │ (None, 224, 224,  │          0 │ image[0][0]       │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_block1… │ (None, 14, 14,    │     98,512 │ normalize_uint8_… │
│ (Functional)        │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ finder_sepconv      │ (None, 14, 14,    │     20,160 │ mobilenetv2_bloc… │
│ (SeparableConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ finder_bn           │ (None, 14, 14,    │        384 │ finder_sepconv[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ finder_relu6 (ReLU) │ (None, 14, 14,    │          0 │ finder_bn[0][0]   │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ box_logits_conv     │ (None, 14, 14,    │      1,164 │ finder_relu6[0][… │
│ (Conv2D)            │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_logits_conv   │ (None, 14, 14, 3) │        291 │ finder_relu6[0][… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ box_output          │ (None, 14, 14, 3, │          0 │ box_logits_conv[… │
│ (Reshape)           │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_output        │ (None, 14, 14, 3, │          0 │ score_logits_con… │
│ (Reshape)           │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 120,511 (470.75 KB)

 Trainable params: 113,775 (444.43 KB)

 Non-trainable params: 6,736 (26.31 KB)

## Train or load compatible weights

In [ ]:
train_dataset = make_dataset(
    datasets["train_crop"], anchors, GRID_SIZE, INPUT_SIZE, BATCH_SIZE, training=True
)
validation_dataset = make_dataset(
    datasets["validation_crop"], anchors, GRID_SIZE, INPUT_SIZE, BATCH_SIZE, training=False
)


def callbacks(stage: str, append: bool) -> list[Any]:
    return [
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(checkpoint_dir / f"{stage}_backup")
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=base.EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(
            str(artifact_dir / "training_log.csv"), append=append
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


if paths.final_weights.is_file():
    model.load_weights(paths.final_weights)
    print("Loaded existing compatible final weights:", paths.final_weights)
elif not RUN_TRAINING:
    raise FileNotFoundError(
        f"RUN_TRAINING=False but compatible weights do not exist: {paths.final_weights}"
    )
else:
    if paths.head_weights.is_file():
        model.load_weights(paths.head_weights)
        print("Resuming from head weights:", paths.head_weights)
    else:
        feature_extractor.trainable = False
        compile_model(model, anchors, GRID_SIZE, base.HEAD_LEARNING_RATE)
        model.fit(
            train_dataset,
            validation_data=validation_dataset,
            epochs=base.HEAD_EPOCHS,
            callbacks=callbacks("head", False),
        )
        model.save_weights(paths.head_weights)

    feature_extractor.trainable = True
    for layer in feature_extractor.layers[:-base.FINE_TUNE_BACKBONE_LAYERS]:
        layer.trainable = False
    for layer in feature_extractor.layers[-base.FINE_TUNE_BACKBONE_LAYERS:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, anchors, GRID_SIZE, base.FINE_TUNE_LEARNING_RATE)
    model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=base.FINE_TUNE_EPOCHS,
        callbacks=callbacks("fine_tune", True),
    )
    model.save_weights(paths.final_weights)
    print("Saved final weights:", paths.final_weights)

# Compile state is not needed for export, but weights must always be explicit.
model.load_weights(paths.final_weights)


Epoch 1/8


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793055487.442492    3817 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1793055488.421957    3728 service.cc:153] XLA service 0x723b000333f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793055488.421993    3728 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793055488.564699    3728 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1

      3/Unknown 16s 45ms/step - box_output_loss: 10.4741 - loss: 50.5465 - score_output_loss: 40.0723 - score_output_mean_ignored_score: 0.4726 - score_output_mean_negative_score: 0.4551 - score_output_mean_positive_score: 0.6257

I0000 00:00:1793055499.119071    3728 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   1585/Unknown 66s 32ms/step - box_output_loss: 0.5301 - loss: 2.2414 - score_output_loss: 1.7111 - score_output_mean_ignored_score: 0.2463 - score_output_mean_negative_score: 0.0578 - score_output_mean_positive_score: 0.4878

/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
I0000 00:00:1793055559.406887    4995 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 2991613304150542605


... (13 outputs left out)

I0000 00:00:1793056048.051107    5701 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 2991613304150542605
I0000 00:00:1793056048.051222    5701 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 16467413769622761008
I0000 00:00:1793056048.051268    5701 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 9181266898659020654


1585/1585 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - box_output_loss: 0.1475 - loss: 0.4451 - score_output_loss: 0.2978 - score_output_mean_ignored_score: 0.2588 - score_output_mean_negative_score: 0.0096 - score_output_mean_positive_score: 0.8005 - val_box_output_loss: 0.1652 - val_loss: 1.3207 - val_score_output_loss: 1.1404 - val_score_output_mean_ignored_score: 0.1942 - val_score_output_mean_negative_score: 0.0116 - val_score_output_mean_positive_score: 0.4591 - learning_rate: 3.0000e-05
Epoch 10/32
1585/1585 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - box_output_loss: 0.1455 - loss: 0.4324 - score_output_loss: 0.2869 - score_output_mean_ignored_score: 0.2584 - score_output_mean_negative_score: 0.0095 - score_output_mean_positive_score: 0.8060 - val_box_output_loss: 0.1697 - val_loss: 1.3847 - val_score_output_loss: 1.1989 - val_score_output_mean_ignored_score: 0.1822 - val_score_output_mean_negative_score: 0.0104 - val_score_output_mean_positive_score: 0.4379 - learning_rate: 3.0000e-05
Epoch 

I0000 00:00:1793056213.655566    5701 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 2991613304150542605
I0000 00:00:1793056213.655725    5701 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 16467413769622761008


1585/1585 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - box_output_loss: 0.1431 - loss: 0.4080 - score_output_loss: 0.2649 - score_output_mean_ignored_score: 0.2610 - score_output_mean_negative_score: 0.0090 - score_output_mean_positive_score: 0.8170 - val_box_output_loss: 0.1676 - val_loss: 1.3132 - val_score_output_loss: 1.1308 - val_score_output_mean_ignored_score: 0.2138 - val_score_output_mean_negative_score: 0.0134 - val_score_output_mean_positive_score: 0.4815 - learning_rate: 9.0000e-06
Epoch 16/32
1585/1585 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - box_output_loss: 0.1421 - loss: 0.4061 - score_output_loss: 0.2642 - score_output_mean_ignored_score: 0.2601 - score_output_mean_negative_score: 0.0090 - score_output_mean_positive_score: 0.8166 - val_box_output_loss: 0.1646 - val_loss: 1.3670 - val_score_output_loss: 1.1867 - val_score_output_mean_ignored_score: 0.1970 - val_score_output_mean_negative_score: 0.0111 - val_score_output_mean_positive_score: 0.4622 - learning_rate: 9.0000e-06
Epoch 

## Export float and full-integer UINT8 models

Both branches remain separate model outputs. The converter therefore derives independent quantization parameters for the 4-channel box tensor and the 1-channel score tensor.


In [ ]:
def representative_dataset() -> Iterator[list[np.ndarray]]:
    # Calibrate with the exact Pillow letterbox path used by validation and deployment.
    for record in datasets["calibration_crop"][:base.REPRESENTATIVE_IMAGES]:
        image, _, _, _ = base.prepare_record_image(record, INPUT_SIZE)
        yield [image[None].astype(np.float32)]


float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
paths.float_tflite.write_bytes(float_converter.convert())

uint8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
uint8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
uint8_converter.representative_dataset = representative_dataset
uint8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
uint8_converter.inference_input_type = tf.uint8
uint8_converter.inference_output_type = tf.uint8
paths.uint8_tflite.write_bytes(uint8_converter.convert())

export_details = {
    "float": base.interpreter_details(tf, paths.float_tflite),
    "uint8": base.interpreter_details(tf, paths.uint8_tflite),
}


def output_kind(detail: Mapping[str, Any]) -> str:
    last_dimension = int(detail["shape"][-1])
    if last_dimension == 4:
        return "box"
    if last_dimension == 1:
        return "score"
    raise ValueError(f"Unexpected detector output: {detail}")


expected_shapes = {
    "box": [1, GRID_SIZE, GRID_SIZE, NUM_ANCHORS, 4],
    "score": [1, GRID_SIZE, GRID_SIZE, NUM_ANCHORS, 1],
}
for model_name, details in export_details.items():
    if len(details["outputs"]) != 2:
        raise RuntimeError(f"{model_name} export did not preserve two output tensors")
    by_kind = {output_kind(detail): detail for detail in details["outputs"]}
    for kind, expected_shape in expected_shapes.items():
        if by_kind[kind]["shape"] != expected_shape:
            raise RuntimeError(
                f"Unexpected {model_name} {kind} shape: {by_kind[kind]['shape']}"
            )

uint8_input = export_details["uint8"]["inputs"][0]
if uint8_input["dtype"] != "uint8" or uint8_input["quantization"][0] <= 0:
    raise RuntimeError("Full-integer export is missing a quantized UINT8 input")
for detail in export_details["uint8"]["outputs"]:
    if detail["dtype"] != "uint8" or detail["quantization"][0] <= 0:
        raise RuntimeError(f"Full-integer output is not independently quantized: {detail}")

(export_dir / "tflite_tensor_details.json").write_text(
    json.dumps(base.json_safe(export_details), indent=2), encoding="utf-8"
)
display(pd.DataFrame([
    {"model": model_name, "output_position": position, "kind": output_kind(detail), **detail}
    for model_name, details in export_details.items()
    for position, detail in enumerate(details["outputs"])
]))


## Split-output TFLite inference and float-to-UINT8 drift

`compare_prediction_drift` runs over the complete validation crop set. It compares raw box values, raw score logits, sigmoid score probabilities, and each image's maximum bird score.


In [ ]:
def runtime_outputs_by_kind(interpreter: Any) -> dict[str, Mapping[str, Any]]:
    return {output_kind(detail): detail for detail in interpreter.get_output_details()}


def invoke_split_outputs(interpreter: Any, tensor: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    input_detail = interpreter.get_input_details()[0]
    interpreter.set_tensor(
        input_detail["index"],
        base.quantize_array(tensor[None].astype(np.float32), input_detail),
    )
    interpreter.invoke()
    outputs = runtime_outputs_by_kind(interpreter)
    raw_box = base.dequantize_array(
        interpreter.get_tensor(outputs["box"]["index"]), outputs["box"]
    )[0]
    raw_score = base.dequantize_array(
        interpreter.get_tensor(outputs["score"]["index"]), outputs["score"]
    )[0, ..., 0]
    return raw_box, raw_score


def decode_split_output(
    raw_box: np.ndarray,
    raw_score: np.ndarray,
    anchors: np.ndarray,
    input_size: int,
    original_width: int,
    original_height: int,
    scale: float,
    pad_x: float,
    pad_y: float,
) -> tuple[np.ndarray, np.ndarray]:
    grid_height, grid_width = raw_box.shape[:2]
    grid_y, grid_x = np.meshgrid(
        np.arange(grid_height), np.arange(grid_width), indexing="ij"
    )
    offset = (CENTER_DECODE_SPAN_CELLS - 1) / 2
    center_x = (
        base.sigmoid_np(raw_box[..., 0]) * CENTER_DECODE_SPAN_CELLS
        - offset + grid_x[..., None]
    ) / grid_width
    center_y = (
        base.sigmoid_np(raw_box[..., 1]) * CENTER_DECODE_SPAN_CELLS
        - offset + grid_y[..., None]
    ) / grid_height
    width = anchors[None, None, :, 0] * np.exp(np.clip(raw_box[..., 2], -4, 4))
    height = anchors[None, None, :, 1] * np.exp(np.clip(raw_box[..., 3], -4, 4))
    scores = base.sigmoid_np(raw_score)
    selected = scores >= base.PRE_NMS_SCORE
    if not selected.any():
        return np.empty((0, 4), np.float32), np.empty((0,), np.float32)

    boxes = np.stack(
        [
            center_x - width / 2,
            center_y - height / 2,
            center_x + width / 2,
            center_y + height / 2,
        ],
        axis=-1,
    )[selected]
    scores = scores[selected]
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] * input_size - pad_x) / scale
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] * input_size - pad_y) / scale
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, original_width)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, original_height)
    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    boxes, scores = boxes[valid], scores[valid]

    order, keep = np.argsort(scores)[::-1], []
    while len(order) and len(keep) < base.MAX_DETECTIONS:
        current = int(order[0])
        keep.append(current)
        if len(order) == 1:
            break
        remaining = order[1:]
        order = remaining[
            base.box_iou_array(boxes[current], boxes[remaining]) < base.NMS_IOU
        ]
    keep_array = np.asarray(keep, np.int32)
    return boxes[keep_array], scores[keep_array]


def predict_tflite(
    model_path: Path,
    records: Sequence[Mapping[str, Any]],
    anchors: np.ndarray,
    input_size: int,
) -> pd.DataFrame:
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    predictions: list[dict[str, Any]] = []
    for record in records:
        tensor, scale, pad_x, pad_y = base.prepare_record_image(record, input_size)
        raw_box, raw_score = invoke_split_outputs(interpreter, tensor)
        boxes, scores = decode_split_output(
            raw_box,
            raw_score,
            anchors,
            input_size,
            record["width"],
            record["height"],
            scale,
            pad_x,
            pad_y,
        )
        predictions.extend({
            "sample_id": record["sample_id"],
            "xmin": float(box[0]),
            "ymin": float(box[1]),
            "xmax": float(box[2]),
            "ymax": float(box[3]),
            "score": float(score),
        } for box, score in zip(boxes, scores))
    return pd.DataFrame(
        predictions, columns=["sample_id", "xmin", "ymin", "xmax", "ymax", "score"]
    )


def compare_prediction_drift(
    float_model_path: Path,
    uint8_model_path: Path,
    records: Sequence[Mapping[str, Any]],
    input_size: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare dequantized full-integer outputs with float outputs on every record."""
    float_interpreter = tf.lite.Interpreter(model_path=str(float_model_path))
    uint8_interpreter = tf.lite.Interpreter(model_path=str(uint8_model_path))
    float_interpreter.allocate_tensors()
    uint8_interpreter.allocate_tensors()

    rows: list[dict[str, Any]] = []
    all_box_drift: list[np.ndarray] = []
    all_logit_drift: list[np.ndarray] = []
    all_probability_drift: list[np.ndarray] = []
    for record in records:
        tensor, _, _, _ = base.prepare_record_image(record, input_size)
        float_box, float_score = invoke_split_outputs(float_interpreter, tensor)
        uint8_box, uint8_score = invoke_split_outputs(uint8_interpreter, tensor)
        box_drift = np.abs(float_box - uint8_box)
        logit_drift = np.abs(float_score - uint8_score)
        probability_drift = np.abs(
            base.sigmoid_np(float_score) - base.sigmoid_np(uint8_score)
        )
        all_box_drift.append(box_drift.reshape(-1))
        all_logit_drift.append(logit_drift.reshape(-1))
        all_probability_drift.append(probability_drift.reshape(-1))
        rows.append({
            "sample_id": record["sample_id"],
            "box_mae": float(box_drift.mean()),
            "box_max_abs": float(box_drift.max()),
            "score_logit_mae": float(logit_drift.mean()),
            "score_logit_max_abs": float(logit_drift.max()),
            "score_probability_mae": float(probability_drift.mean()),
            "score_probability_max_abs": float(probability_drift.max()),
            "float_max_score": float(base.sigmoid_np(float_score).max()),
            "uint8_max_score": float(base.sigmoid_np(uint8_score).max()),
        })

    per_image = pd.DataFrame(rows)
    box_values = np.concatenate(all_box_drift) if all_box_drift else np.asarray([], np.float32)
    logit_values = np.concatenate(all_logit_drift) if all_logit_drift else np.asarray([], np.float32)
    probability_values = (
        np.concatenate(all_probability_drift)
        if all_probability_drift else np.asarray([], np.float32)
    )

    def summary_row(name: str, values: np.ndarray) -> dict[str, Any]:
        return {
            "quantity": name,
            "count": int(values.size),
            "mean_abs_drift": float(values.mean()) if values.size else 0.0,
            "p95_abs_drift": float(np.quantile(values, 0.95)) if values.size else 0.0,
            "max_abs_drift": float(values.max()) if values.size else 0.0,
        }

    summary = pd.DataFrame([
        summary_row("box_raw", box_values),
        summary_row("score_logit", logit_values),
        summary_row("score_probability", probability_values),
    ])
    return per_image, summary


drift_per_image, drift_summary = compare_prediction_drift(
    paths.float_tflite,
    paths.uint8_tflite,
    datasets["validation_crop"],
    INPUT_SIZE,
)
drift_per_image.to_csv(artifact_dir / "validation_float_uint8_drift_per_image.csv", index=False)
drift_summary.to_csv(artifact_dir / "validation_float_uint8_drift_summary.csv", index=False)
drift_limits = {
    "score_probability_mean_abs_drift": 0.02,
    "score_probability_p95_abs_drift": 0.05,
    "max_score_mean_abs_drift": 0.05,
}
score_probability_summary = drift_summary.loc[
    drift_summary.quantity.eq("score_probability")
].iloc[0]
max_score_mean_abs_drift = float(
    np.abs(drift_per_image.float_max_score - drift_per_image.uint8_max_score).mean()
)
drift_passed = bool(
    score_probability_summary.mean_abs_drift
    <= drift_limits["score_probability_mean_abs_drift"]
    and score_probability_summary.p95_abs_drift
    <= drift_limits["score_probability_p95_abs_drift"]
    and max_score_mean_abs_drift
    <= drift_limits["max_score_mean_abs_drift"]
)
display(drift_summary)
print("Mean absolute drift of each image's maximum score:", max_score_mean_abs_drift)
print("Drift acceptance passed:", drift_passed)


## Select the deployment threshold on the UINT8 validation sweep

The float sweep is saved only as a diagnostic. It does not choose the deployment threshold. Both float and UINT8 metrics below use the threshold selected by the UINT8 model itself.


In [ ]:
float_validation_predictions = predict_tflite(
    paths.float_tflite, datasets["validation_crop"], anchors, INPUT_SIZE
)
uint8_validation_predictions = predict_tflite(
    paths.uint8_tflite, datasets["validation_crop"], anchors, INPUT_SIZE
)

uint8_threshold, uint8_sweep = base.select_threshold(
    datasets["validation_crop"], uint8_validation_predictions
)
feasible_uint8_sweep = uint8_sweep.loc[
    uint8_sweep.false_positives_per_negative_image
    <= base.MAX_FALSE_POSITIVES_PER_NEGATIVE_CROP
]
if feasible_uint8_sweep.empty:
    raise RuntimeError(
        "No UINT8 validation threshold satisfies the false-positive constraint; "
        "the model is not deployable under the configured policy."
    )
selected_uint8_row = uint8_sweep.loc[
    np.isclose(uint8_sweep.confidence_threshold, uint8_threshold)
].iloc[0]
false_positive_constraint_satisfied = bool(
    selected_uint8_row.false_positives_per_negative_image
    <= base.MAX_FALSE_POSITIVES_PER_NEGATIVE_CROP
)
float_diagnostic_sweep = pd.DataFrame([
    base.evaluate_predictions(
        datasets["validation_crop"], float_validation_predictions, threshold
    )
    for threshold in base.CONFIDENCE_GRID
])

uint8_sweep.to_csv(artifact_dir / "uint8_validation_crop_threshold_sweep.csv", index=False)
float_diagnostic_sweep.to_csv(
    artifact_dir / "float_validation_crop_threshold_sweep_diagnostic.csv", index=False
)

metric_rows: list[dict[str, Any]] = []
validation_predictions = {
    "float": float_validation_predictions,
    "uint8": uint8_validation_predictions,
}
for model_name, model_path in (
    ("float", paths.float_tflite),
    ("uint8", paths.uint8_tflite),
):
    for dataset_name in ("validation_crop", "test_crop"):
        records = datasets[dataset_name]
        predictions = (
            validation_predictions[model_name]
            if dataset_name == "validation_crop"
            else predict_tflite(model_path, records, anchors, INPUT_SIZE)
        )
        metric_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "threshold_source": "uint8_validation_sweep",
            **base.evaluate_predictions(records, predictions, uint8_threshold),
        })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(artifact_dir / "crop_metrics_at_uint8_threshold.csv", index=False)
print("Deployment threshold selected by UINT8 validation sweep:", uint8_threshold)
display(metrics)


## Write split-output decoder metadata and run contract

In [ ]:
uint8_outputs = {
    output_kind(detail): {**detail, "output_position": position}
    for position, detail in enumerate(export_details["uint8"]["outputs"])
}
score_scale, score_zero_point = uint8_outputs["score"]["quantization"]
raw_score_values = np.arange(256, dtype=np.uint8)
raw_score_probabilities = base.sigmoid_np(
    (raw_score_values.astype(np.float32) - score_zero_point) * score_scale
)
eligible_raw_scores = raw_score_values[raw_score_probabilities >= uint8_threshold]
uint8_raw_score_cutoff = int(eligible_raw_scores[0]) if len(eligible_raw_scores) else None
decoder_metadata = {
    "schema_version": 3,
    "model": paths.uint8_tflite.name,
    "model_sha256": base.sha256_file(paths.uint8_tflite),
    "class_map": {"0": "bird"},
    "input": {
        **export_details["uint8"]["inputs"][0],
        "color_order": "RGB",
        "geometry": "selected_crop_then_letterbox",
        "letterbox_padding_value": 114,
        "numeric_scaling": "[0,255] to [-1,1] inside the model",
    },
    "outputs": {
        "box": {
            **uint8_outputs["box"],
            "order": ["tx", "ty", "tw", "th"],
        },
        "score": {
            **uint8_outputs["score"],
            "order": ["bird_score_logit"],
            "activation": "sigmoid outside the model",
        },
        "anchors_normalized_wh": anchors.tolist(),
        "decode": {
            "dequantize_each_output": "real_value = (quantized_value - zero_point) * scale",
            "center_x": "(sigmoid(tx) * 3 - 1 + grid_x) / grid_width",
            "center_y": "(sigmoid(ty) * 3 - 1 + grid_y) / grid_height",
            "width": "anchor_width * exp(clamp(tw, -4, 4))",
            "height": "anchor_height * exp(clamp(th, -4, 4))",
            "score": "sigmoid(bird_score_logit)",
        },
    },
    "postprocess": {
        "candidate_score_threshold": base.PRE_NMS_SCORE,
        "deployment_score_threshold": uint8_threshold,
        "deployment_threshold_source": "UINT8 validation motion-crop sweep",
        "uint8_raw_score_cutoff": uint8_raw_score_cutoff,
        "uint8_raw_score_cutoff_semantics": (
            "minimum raw UINT8 score-logit tensor value whose dequantized sigmoid is at least "
            "deployment_score_threshold"
        ),
        "nms_iou": base.NMS_IOU,
        "max_detections": base.MAX_DETECTIONS,
    },
}
metadata_path = export_dir / "bird_detector_motion_crop_stride16_metadata.json"
metadata_path.write_text(
    json.dumps(base.json_safe(decoder_metadata), indent=2, allow_nan=False),
    encoding="utf-8",
)

run_contract = {
    "schema_version": 3,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_run": str(source_run),
    "task": "single-class bird detection on Pi-selected motion crops",
    "architecture": {
        "backbone": "MobileNetV2 ImageNet transfer learning",
        "backbone_alpha": BACKBONE_ALPHA,
        "feature_layer": FEATURE_LAYER,
        "feature_stride": FEATURE_STRIDE,
        "input_size": INPUT_SIZE,
        "grid_size": GRID_SIZE,
        "anchors_normalized_wh": anchors.tolist(),
        "head": "separate box and bird-score 1x1 convolutions",
        "output_shapes": expected_shapes,
    },
    "target_encoding": {
        "channels": ["cx", "cy", "width", "height", "objectness", "ignore", "bird_count"],
        "ignore_policy": "non-positive slots ignored when their cell centre is inside any bird box",
    },
    "training_loss": {
        "box": "positive slots only; Huber; normalized by unique birds in batch",
        "positive_score": "focal BCE; normalized by unique birds in batch",
        "negative_score": "focal BCE on (1-positive)*(1-ignore); normalized by unique birds in batch",
        "ignored_slots_in_negative_term": False,
    },
    "anchor_policy": {
        "re_fitted_for_this_configuration": bool(REFIT_ANCHORS),
        "fit_source": "normalized letterboxed train_crop box widths/heights",
    },
    "evaluation_policy": {
        "threshold_selection_dataset": "validation_crop",
        "threshold_selection_model": "uint8",
        "selected_threshold": uint8_threshold,
        "false_positive_constraint_satisfied": false_positive_constraint_satisfied,
        "selected_false_positives_per_negative_image": float(
            selected_uint8_row.false_positives_per_negative_image
        ),
        "maximum_allowed_false_positives_per_negative_image": (
            base.MAX_FALSE_POSITIVES_PER_NEGATIVE_CROP
        ),
        "float_sweep_role": "diagnostic only",
        "float_uint8_drift_dataset": "validation_crop",
    },
    "drift_acceptance": {
        "passed": drift_passed,
        "limits": drift_limits,
        "max_score_mean_abs_drift": max_score_mean_abs_drift,
    },
    "drift_summary": drift_summary.to_dict(orient="records"),
    "metrics_at_uint8_threshold": metrics.to_dict(orient="records"),
    "exports": export_details,
}
(artifact_dir / "run_contract.json").write_text(
    json.dumps(base.json_safe(run_contract), indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Artifacts:", artifact_dir)
print("UINT8 TFLite:", paths.uint8_tflite)
print("Decoder metadata:", metadata_path)


## Inspect training history

The ignored-score metric is diagnostic only: ignored slots are explicitly absent from the negative loss.


In [ ]:
training_log = artifact_dir / "training_log.csv"
if training_log.is_file():
    history = pd.read_csv(training_log)
    display(history.tail())
    score_columns = [column for column in history.columns if "score" in column]
    if score_columns:
        history.plot(x="epoch", y=score_columns, grid=True, figsize=(11, 5))
else:
    print("No training log yet:", training_log)
